Week 4-5 Deliverable: Data Cleaning and Preparation -- LISTING dataset

"""
1. Converts date fields to datetime format
2. Removes unnecessary/redundant columns (>90% null, from Week 2-3 report)
3. Ensures numeric fields are properly typed
4. Flags invalid numeric values (ClosePrice<=0, LivingArea<=0, DaysOnMarket<0,
   negative Bedrooms/Bathrooms)
5. Runs date consistency checks (listing_after_close_flag,
   purchase_after_close_flag, negative_timeline_flag)
6. Runs geographic data checks (missing/sentinel/wrong-sign coordinates)
7. Adds school district mapping (Aidan's Slack post) via spatial join
8. Saves a cleaned, analysis-ready CSV
"""

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
 
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
 
DATE_FIELDS = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']

In [ ]:
# Columns with >90% missing values found in the Week 2-3 EDA report.
HIGH_NULL_COLUMNS_TO_DROP = [
    'MiddleOrJuniorSchoolDistrict',
    'BusinessType',
    'TaxYear',
    'ElementarySchoolDistrict',
    'TaxAnnualAmount',
    'FireplacesTotal',
    'AboveGradeFinishedArea',
]
 
NUMERIC_FIELDS = ['ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea',
                   'LotSizeAcres', 'BedroomsTotal', 'BathroomsTotalInteger',
                   'DaysOnMarket', 'YearBuilt']
 
# School district GeoJSON  
# data.ca.gov/dataset/california-school-district-areas-2025-26
GEOJSON_PATH = 'california_school_district_areas_2025_26.geojson'

In [3]:
# PART - LISTING DATASET
# =================================================================
print("=" * 70)
print("CLEANING: LISTING  (listing_with_rates.csv)")
print("=" * 70)
df_listing = pd.read_csv('listing_with_rates.csv', low_memory=False)
rows_before_listing = len(df_listing)
cols_before_listing = len(df_listing.columns)
print(f"\nLoaded {rows_before_listing} rows, {cols_before_listing} columns")

CLEANING: LISTING  (listing_with_rates.csv)

Loaded 614541 rows, 75 columns


In [5]:
# --- 1. Convert date fields to datetime ---
df_listing['CloseDate'] = pd.to_datetime(df_listing['CloseDate'], errors='coerce')
df_listing['PurchaseContractDate'] = pd.to_datetime(df_listing['PurchaseContractDate'], errors='coerce')
df_listing['ListingContractDate'] = pd.to_datetime(df_listing['ListingContractDate'], errors='coerce')
df_listing['ContractStatusChangeDate'] = pd.to_datetime(df_listing['ContractStatusChangeDate'], errors='coerce')

In [6]:
# --- 2. Remove high-null / redundant columns ---
print("\n--- 2. Removing high-null / redundant columns ---")
cols_present_listing = [c for c in HIGH_NULL_COLUMNS_TO_DROP if c in df_listing.columns]
cols_before_drop_listing = len(df_listing.columns)
df_listing = df_listing.drop(columns=cols_present_listing)
 
print(f"  Dropped {len(cols_present_listing)} columns: {cols_present_listing}")
print(f"  Columns before: {cols_before_drop_listing}, after: {len(df_listing.columns)}")


--- 2. Removing high-null / redundant columns ---
  Dropped 7 columns: ['MiddleOrJuniorSchoolDistrict', 'BusinessType', 'TaxYear', 'ElementarySchoolDistrict', 'TaxAnnualAmount', 'FireplacesTotal', 'AboveGradeFinishedArea']
  Columns before: 75, after: 68


In [7]:
# --- 3. Ensure numeric fields are properly typed ---
print("\n--- 3. Ensuring numeric fields are properly typed ---")
for col in NUMERIC_FIELDS:
    if col in df_listing.columns:
        before_dtype = df_listing[col].dtype
        df_listing[col] = pd.to_numeric(df_listing[col], errors='coerce')
        print(f"  {col}: {before_dtype} -> {df_listing[col].dtype}")


--- 3. Ensuring numeric fields are properly typed ---
  ClosePrice: float64 -> float64
  ListPrice: float64 -> float64
  OriginalListPrice: float64 -> float64
  LivingArea: float64 -> float64
  LotSizeAcres: float64 -> float64
  BedroomsTotal: float64 -> float64
  BathroomsTotalInteger: float64 -> float64
  DaysOnMarket: int64 -> int64
  YearBuilt: float64 -> float64


In [8]:
# --- 4. Flag invalid numeric values ---
print("\n--- 4. Flagging invalid numeric values ---")
df_listing['invalid_close_price_flag'] = df_listing['ClosePrice'] <= 0 if 'ClosePrice' in df_listing.columns else False
df_listing['invalid_living_area_flag'] = df_listing['LivingArea'] <= 0 if 'LivingArea' in df_listing.columns else False
df_listing['invalid_days_on_market_flag'] = df_listing['DaysOnMarket'] < 0 if 'DaysOnMarket' in df_listing.columns else False
df_listing['invalid_bedrooms_flag'] = df_listing['BedroomsTotal'] < 0 if 'BedroomsTotal' in df_listing.columns else False
df_listing['invalid_bathrooms_flag'] = df_listing['BathroomsTotalInteger'] < 0 if 'BathroomsTotalInteger' in df_listing.columns else False
 
print(f"  invalid_close_price_flag:    {df_listing['invalid_close_price_flag'].sum()} records")
print(f"  invalid_living_area_flag:    {df_listing['invalid_living_area_flag'].sum()} records")
print(f"  invalid_days_on_market_flag: {df_listing['invalid_days_on_market_flag'].sum()} records")
print(f"  invalid_bedrooms_flag:       {df_listing['invalid_bedrooms_flag'].sum()} records")
print(f"  invalid_bathrooms_flag:      {df_listing['invalid_bathrooms_flag'].sum()} records")


--- 4. Flagging invalid numeric values ---
  invalid_close_price_flag:    0 records
  invalid_living_area_flag:    393 records
  invalid_days_on_market_flag: 31 records
  invalid_bedrooms_flag:       0 records
  invalid_bathrooms_flag:      0 records


In [9]:
# --- 5. Date consistency checks ---
print("\n--- 5. Date consistency checks ---")
has_all_date_fields_listing = all(c in df_listing.columns for c in ['ListingContractDate', 'PurchaseContractDate', 'CloseDate'])
 
if has_all_date_fields_listing:
    df_listing['listing_after_close_flag'] = df_listing['ListingContractDate'] > df_listing['CloseDate']
    df_listing['purchase_after_close_flag'] = df_listing['PurchaseContractDate'] > df_listing['CloseDate']
    df_listing['negative_timeline_flag'] = (
        df_listing['listing_after_close_flag']
        | df_listing['purchase_after_close_flag']
        | (df_listing['ListingContractDate'] > df_listing['PurchaseContractDate'])
    )
 
    print(f"  listing_after_close_flag:  {df_listing['listing_after_close_flag'].sum()} records "
          f"(ListingContractDate is after CloseDate)")
    print(f"  purchase_after_close_flag: {df_listing['purchase_after_close_flag'].sum()} records "
          f"(PurchaseContractDate is after CloseDate)")
    print(f"  negative_timeline_flag:    {df_listing['negative_timeline_flag'].sum()} records "
          f"(any date-order violation)")
else:
    print("  Missing one or more of ListingContractDate/PurchaseContractDate/CloseDate, skipping.")
    df_listing['listing_after_close_flag'] = False
    df_listing['purchase_after_close_flag'] = False
    df_listing['negative_timeline_flag'] = False


--- 5. Date consistency checks ---
  listing_after_close_flag:  84 records (ListingContractDate is after CloseDate)
  purchase_after_close_flag: 266 records (PurchaseContractDate is after CloseDate)
  negative_timeline_flag:    566 records (any date-order violation)


In [10]:
# --- 6. Geographic data checks ---
print("\n--- 6. Geographic data checks ---")
has_coords_listing = 'Latitude' in df_listing.columns and 'Longitude' in df_listing.columns
 
if has_coords_listing:
    df_listing['Latitude'] = pd.to_numeric(df_listing['Latitude'], errors='coerce')
    df_listing['Longitude'] = pd.to_numeric(df_listing['Longitude'], errors='coerce')
 
    df_listing['missing_coords_flag'] = df_listing['Latitude'].isnull() | df_listing['Longitude'].isnull()
    df_listing['sentinel_zero_coords_flag'] = (df_listing['Latitude'] == 0) | (df_listing['Longitude'] == 0)
    df_listing['wrong_sign_longitude_flag'] = df_listing['Longitude'] > 0
 
    df_listing['implausible_coords_flag'] = (
        ~df_listing['missing_coords_flag'] &
        ((df_listing['Latitude'] < 32) | (df_listing['Latitude'] > 42) |
         (df_listing['Longitude'] < -125) | (df_listing['Longitude'] > -114))
    )
 
    print(f"  missing_coords_flag:        {df_listing['missing_coords_flag'].sum()} records")
    print(f"  sentinel_zero_coords_flag:  {df_listing['sentinel_zero_coords_flag'].sum()} records")
    print(f"  wrong_sign_longitude_flag:  {df_listing['wrong_sign_longitude_flag'].sum()} records")
    print(f"  implausible_coords_flag:    {df_listing['implausible_coords_flag'].sum()} records "
          f"(outside rough CA lat/lon bounding box)")
else:
    print("  Latitude/Longitude not present, skipping.")
    df_listing['missing_coords_flag'] = False
    df_listing['sentinel_zero_coords_flag'] = False
    df_listing['wrong_sign_longitude_flag'] = False
    df_listing['implausible_coords_flag'] = False


--- 6. Geographic data checks ---
  missing_coords_flag:        80976 records
  sentinel_zero_coords_flag:  74 records
  wrong_sign_longitude_flag:  83 records
  implausible_coords_flag:    320 records (outside rough CA lat/lon bounding box)


In [11]:
# --- 7. Add school district mapping  ---
print("\n--- 7. Adding school district mapping ---")
 
districts = gpd.read_file(GEOJSON_PATH)
districts_unified = districts[districts['DistrictType'] == 'Unified'].copy()
print(f"  Loaded {len(districts)} district polygons, filtered to {len(districts_unified)} Unified districts")
 
# Only rows with usable coordinates can be spatially joined
join_mask_listing = df_listing['Latitude'].notnull() & df_listing['Longitude'].notnull()
df_for_join_listing = df_listing[join_mask_listing].copy()
 
geometry_listing = [Point(lon, lat) for lon, lat in
                  zip(df_for_join_listing['Longitude'], df_for_join_listing['Latitude'])]
gdf_listing = gpd.GeoDataFrame(df_for_join_listing, geometry=geometry_listing, crs='EPSG:4326')
 
if districts_unified.crs != gdf_listing.crs:
    districts_unified = districts_unified.to_crs(gdf_listing.crs)
 
joined_listing = gpd.sjoin(
    gdf_listing,
    districts_unified[['DistrictName', 'geometry']],
    how='left',
    predicate='within'
)
 
# Merge DistrictName back onto the full df_listing by index, so rows with
# missing coordinates are preserved (just with DistrictName = NaN)
df_listing = df_listing.merge(joined_listing[['DistrictName']], left_index=True, right_index=True, how='left')
 
print(f"  Matched to a Unified school district: {df_listing['DistrictName'].notnull().sum()} records")
print(f"  Not matched (missing coords or outside any polygon): {df_listing['DistrictName'].isnull().sum()} records")


--- 7. Adding school district mapping ---
  Loaded 936 district polygons, filtered to 345 Unified districts
  Matched to a Unified school district: 411061 records
  Not matched (missing coords or outside any polygon): 203480 records


In [12]:
# --- 8. Summary + save ---
rows_after_listing = len(df_listing)
cols_after_listing = len(df_listing.columns)
 
print(f"\n--- Summary for listing ---")
print(f"Rows before: {rows_before_listing}, Rows after: {rows_after_listing} (no rows dropped, only flagged/enriched)")
print(f"Columns before: {cols_before_listing}, Columns after: {cols_after_listing}")
 
df_listing.to_csv('listing_cleaned.csv', index=False, encoding='utf-8')
print(f"\nSaved cleaned dataset: listing_cleaned.csv ({rows_after_listing} rows, {cols_after_listing} columns)")


--- Summary for listing ---
Rows before: 614541, Rows after: 614541 (no rows dropped, only flagged/enriched)
Columns before: 75, Columns after: 81

Saved cleaned dataset: listing_cleaned.csv (614541 rows, 81 columns)
